In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
# disable warnings
import warnings
warnings.filterwarnings('ignore')

# import functions from my_path\phd work major comparison\utils.py
# use absolute path to import functions
import sys
sys.path.append(r'my_path/phd work major comparison')
from utils import *

In [2]:
# regression

import stata_setup

stata_setup.config("C:/Program Files/Stata17/", "se")
from pystata import stata


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      17.0
___/   /   /___/   /   /___/       SE—Standard Edition

 Statistics and Data Science       Copyright 1985-2021 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-STATA-PC        https://www.stata.com
                                   979-696-4600        stata@stata.com

Stata license: Unlimited-user network, expiring 23 Oct 2025
Serial number: 401809302077
  Licensed to: Xiang
               UW-Madison

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


In [ ]:
# read ID list
idlist = pd.read_pickle(r"personid_list_2to21.pkl")

# import IDR data and degree year
df_idr = pd.read_csv(r"discipline_idr_bin.csv")
df_idr = df_idr.query("PersonId in @idlist")
df_idr['rao'] = df_idr['rao'].apply(lambda x: 0 if x <= 0 else x)

# institution and degree year
df_inst = pd.read_csv(
    r"my_path\university match\faculty_inst_phdinst.csv"
)

# remove people graduated in 2020
df_inst = df_inst.query("DegreeYear < 2019")
df_inst = df_inst.sort_values(["survey_year"]).drop_duplicates("PersonId", keep="first")

# read distance data
df_dist = pd.read_csv(
    r"my_path\phd work major comparison\10 interdiscipnarity factor\distance factor\discipline_distance_cross.csv"
)
# remove humanities
df_dist = df_dist.query("phd_community != 5")
df_dist

In [4]:
# delete humanities
t = {k: v for k, v in community_names.items() if k != 11}
areas = list(t.keys())
area_names = list(t.values())
print(areas)

[0, 1, 2, 3, 4]


# Graduate year

In [ ]:
df_base = (
    df_idr
    .merge(df_dist, on=["PersonId", "phd_area"], how="inner")
    .merge(df_inst[['PersonId', 'DegreeInstitutionID', "InstitutionId", 'survey_year']], on="PersonId", how="inner")
    .drop_duplicates()
    # remove people graduated in 2020
    .query("DegreeYear < 2019 and DegreeYear >= 2005")
)
df_base

In [6]:
print(df_base['PersonId'].nunique())
df_adv = pd.read_csv(r"advisor_attr.csv")
df_base.query("PersonId in @df_adv.PersonId")['PersonId'].nunique()


32977


29096

In [ ]:
# read gender data
df_gender = pd.read_csv(
    r"my_path\phd work major comparison\5. get gender and race\gender_match_discipline.csv"
)
display(df_gender["gender"].value_counts())

gender
M    31999
F    24358
U     2172
Name: count, dtype: int64

In [8]:
(df_base.merge(df_gender, on="PersonId", how="inner")).groupby(["gender"])[
    "PersonId"
].nunique().reset_index()

,gender,PersonId
0,F,12661
1,M,19268
2,U,1217


In [ ]:
# compare IDR by gender
df_gender_base = (
    df_base
    .merge(df_gender.query("gender != 'U'"), on="PersonId", how="inner")
)
df_gender_base

# Univ rank

In [ ]:
# read race data
df_univ = pd.read_csv(
    r"my_path\phd work major comparison\6. get dept rank\institution_rank_by_area.csv"
)
# change area names in df_univ
# replace full names with short names
df_univ["area"] = df_univ["area"].map(abbr)
display(df_univ)

,Rank,InstitutionId,Percentile,area
0,0.000000,7,0.000000,Sociology
1,0.250711,217,0.440529,Sociology
2,0.547605,539130,0.881057,Sociology
3,0.713267,395,1.321586,Sociology
4,0.714773,388,1.762115,Sociology
...,...,...,...,...
4956,10.501938,49,98.181818,Physics
4957,10.963708,198,98.636364,Physics
4958,11.083249,251,99.090909,Physics
4959,11.207247,167,99.545455,Physics


In [ ]:
# distance by university
df_univ_base = (
    df_gender_base.merge(
        df_univ[["InstitutionId", "Percentile", "area"]].rename(
            columns={"area": "prof_area"}
        ),
        on=["InstitutionId", "prof_area"],
        how="left",
    )
    .merge(
        df_univ.rename(
            columns={
                "InstitutionId": "DegreeInstitutionID",
                "Percentile": "DegreePercentile",
                "area": "phd_area",
            }
        )[["DegreeInstitutionID", "DegreePercentile", "phd_area"]],
        on=["DegreeInstitutionID", "phd_area"],
        how="inner",
    )
    .drop_duplicates()
)

for top in range(5, 21, 5):
    df_univ_base[f"rank{top}"] = df_univ_base["Percentile"].apply(
        lambda x: np.nan if pd.isna(x) else 1 if x >= 100 - top else 0
    )  # can use top 5% or top 10%
    df_univ_base[f"degree_rank{top}"] = df_univ_base["DegreePercentile"].apply(
        lambda x: np.nan if pd.isna(x) else 1 if x >= 100 - top else 0
    )

# cut into quartiles
df_univ_base["quartile"] = pd.qcut(df_univ_base["Percentile"], 4, labels=False)
df_univ_base["degree_quartile"] = pd.qcut(
    df_univ_base["DegreePercentile"], 4, labels=False
)

df_univ_base["Percentile_diff"] = (
    df_univ_base["Percentile"] - df_univ_base["DegreePercentile"]
)
df_univ_base["trend"] = df_univ_base["Percentile_diff"].apply(
    lambda x: 1 if x >= 10 else (-1 if x <= -10 else 0)
)

df_univ_base

In [14]:
orders = ['Top 5%', '90-95th', '80-90th', '50-80th', '0-50th']

df_univ_base['interval'] = pd.cut(df_univ_base['Percentile'], bins=[0, 50, 80, 90, 95, 100], labels=orders[::-1])
df_univ_base['phd_interval'] = pd.cut(df_univ_base['DegreePercentile'], bins=[0, 50, 80, 90, 95, 100], labels=orders[::-1])

# calculate person count
data = df_univ_base.groupby(['phd_interval', 'interval', ]).agg(n=('PersonId', 'nunique')).reset_index()

# calculate percentage
total = data['n'].sum()
t1 = data.groupby('phd_interval').agg(phd_pct=('n', 'sum')).reset_index().sort_values('phd_interval',ascending=False)
t2 = data.groupby('interval').agg(prof_pct=('n', 'sum')).reset_index().sort_values('interval',ascending=False)
t1['phd_pct'] = ((t1['phd_pct'] / total)*100).astype(int)
t2['prof_pct'] = ((t2['prof_pct'] / total)*100).astype(int)
t1['text'] = t1.apply(lambda x: f"{x['phd_interval']}<br>({x['phd_pct']}%)", axis=1)
t2['text'] = t2.apply(lambda x: f"{x['interval']}<br>({x['prof_pct']}%)", axis=1)

# colors
colors = generate_gradient_colors(5, 'Blues_r')
lighter_colors = [rgb2rgba(color, 0.5) for color in colors]

# sankey diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=5,
        thickness=20,
        line=dict(color="black", width=0),
        label= t1['text'].tolist() + t2['text'].tolist(),
        color= colors + colors
    ),
    link=dict(
        arrowlen=15,
        source=data['phd_interval'].map(lambda x: orders.index(x)),
        target=data['interval'].map(lambda x: orders.index(x) + len(orders)),
        value=data['n'],
        color=data['phd_interval'].map(lambda x: lighter_colors[orders.index(x)]),
    ),
    orientation="v",
)])

plot_styling(fig, size=(250, 600))

# add column names
fig.add_annotation(x=0.5, y=1.15, text="<b>Faculty's PhD university percentile rank", showarrow=False, font=dict(size=12, color="black"))
fig.add_annotation(x=0.5, y=-0.2, text="<b>Faculty's university percentile rank", showarrow=False, font=dict(size=12, color="black"))

fig.show()
fig.write_image("faculty_phd_percentile_rank_2.pdf")

In [ ]:
# share of top universities by year and field
rs = [5, 10, 15, 20]
ts = []
for r in rs:
    rank = f"rank{r}"
    df = df_univ_base.query("DegreeYear >= 2005 and DegreeYear < 2019")

    t = (
        df.query(f"{rank} == 1")
        .groupby(["phd_community", "DegreeYear"])
        .agg(n=("PersonId", "nunique"))
        .reset_index()
    )
    # total
    total = (
        df.groupby(["phd_community", "DegreeYear"])
        .agg(total=("PersonId", "nunique"))
        .reset_index()
    )
    t = t.merge(total, on=["phd_community", "DegreeYear"])
    t["pct"] = t["n"] / t["total"]
    t["rank"] = rank

    # bootstrap confidence interval
    boots = []
    for seed in range(100):
        df_boot = (
            df.groupby(["phd_community", "DegreeYear"])
            .apply(lambda x: x.sample(frac=1, replace=True, random_state=seed))
            .reset_index(drop=True)
        )
        t_boot = (
            df_boot.query(f"{rank} == 1")
            .groupby(["phd_community", "DegreeYear"])
            .agg(n=("PersonId", "nunique"))
            .reset_index()
        )
        total_boot = (
            df_boot.groupby(["phd_community", "DegreeYear"])
            .agg(total=("PersonId", "nunique"))
            .reset_index()
        )
        t_boot = t_boot.merge(total_boot, on=["phd_community", "DegreeYear"])
        t_boot["pct"] = t_boot["n"] / t_boot["total"]
        t_boot["seed"] = seed
        boots.append(t_boot)

    # get confidence interval
    ci = (
        pd.concat(boots)
        .groupby(["phd_community", "DegreeYear"])
        .agg(
            lower=("pct", lambda x: np.percentile(x, 2.5)),
            upper=("pct", lambda x: np.percentile(x, 97.5)),
        )
        .reset_index()
    )

    t = t.merge(ci, on=["phd_community", "DegreeYear"])
    ts.append(t)

data = pd.concat(ts)
data

In [46]:
# plot line chart
# Create subplots with facets for 'phd_community' and 'inst_type'
fig = make_subplots(
    rows=2, 
    cols=len(data['phd_community'].unique()), 
    subplot_titles=list(community_names.values()),
    shared_xaxes=False,
    shared_yaxes=True,
)

colors = generate_gradient_colors(4, 'Blues_r')

# Plot lines for each combination of 'rank' and facet columns
for community_index, phd_community in enumerate(community_names.keys()):
    for r, rank_value in enumerate([5, 10, 15, 20]):
            filtered_data = data[
                (data['rank'] == f'rank{rank_value}') & 
                (data['phd_community'] == phd_community)
            ]
            fig = add_lines_with_errorband(
                fig,
                x=filtered_data['DegreeYear'],
                y=filtered_data['pct'],
                upper=filtered_data['upper'],
                lower=filtered_data['lower'],
                name=f"Top {rank_value}%",
                color=colors[r],
                showlegend=True if community_index==0 else False,
                row=1,
                col=community_index+1
            )

# Update axis labels and layout
plot_styling(fig, size=[450, 1000])
fig.update_layout(legend=dict(title="Faculty university"))
fig.update_xaxes(range=[2005, 2017])
fig.update_xaxes(title_text="PhD graduation year", col=3)
fig.update_yaxes(tickformat=".0%")
fig.update_yaxes(title_text=r"% faculty placed at<br>top universities", col=1)

fig.show()

In [ ]:
# quantile regression

# fit by phd_community
ts = []
for area in list(community_names.keys()):
    for q in np.arange(0.01, 1, 0.01):
        print(area, community_names[area])
        t = data.query("phd_community == @area")[
            [
                "PersonId",
                "Percentile",
                "n",
                "rao",
                "gender",
                "DegreeYear",
                "DegreePercentile",
                "cite_norm",
                'adv_gender',
                'seniority',
                'adv_n',
                'collab_num'
            ]
        ].drop_duplicates()
        
        # use 0.1 rao as unit
        t['rao'] = t['rao'] * 10
        
        command = f"""
qui qreg Percentile rao DegreePercentile gender cite_norm n i.DegreeYear adv_gender i.seniority adv_n collab_num, vce(robust) quantile({q}) nolog
"""

        stata.run("clear")
        stata.pdataframe_to_data(t)
        stata.run(command)

        # collect results
        results = stata.get_return()["r(table)"].T
        results = pd.DataFrame(results).iloc[0:1, :6]
        results.columns = ["coef", "se", "z", "p", "ci_low", "ci_high"]
        results["phd_community"] = area
        results["quantile"] = q
        ts.append(results)
        
ts = pd.concat(ts)
ts

In [55]:
# plot
fig = make_subplots(
    rows=1,
    cols=5,
    shared_xaxes=False,
    shared_yaxes=False,
    subplot_titles=list(community_names.values()),
)

colors = ['rgb(0, 0, 0)', 'rgb(0, 0, 255)', 'rgb(255, 0, 0)', ]

for i, area in enumerate(list(community_names.keys())):
    for p in ["Insignificant", "Significantly positive", "Significantly negative"]:
        if p == "Insignificant":
            query = "p >= 0.05"
        elif p == "Significantly positive":
            query = "p < 0.05 and coef > 0"
        else:
            query = "p < 0.05 and coef < 0"
        t = ts.query(f"phd_community == @area and {query} and quantile > 0.1")
        color = colors[0] if p == "Insignificant" else colors[1] if p == "Significantly positive" else colors[2]
        error = (t["ci_high"] - t["ci_low"]) / 2
        fig.add_trace(
            go.Scatter(
                x=t["quantile"],
                y=t["coef"],
                error_y=dict(type="data", array=error, width=0, thickness=0.1),
                mode="markers",
                marker=dict(color=color, size=1, opacity=1),
                name=p,
                showlegend=True if i == 4 else False,
            ),
            row=1,
            col=i % 5 + 1,
        )

plot_styling(fig, size=[240, 900])
fig.update_xaxes(title_text="Quantile of faculty university rank", col=3)
fig.update_xaxes(dtick=0.2)
fig.update_yaxes(title_text="Coefficient of<br>interdisciplinarity", row=1, col=1)

# ref line
fig.add_hline(y=0, line=dict(color="black", width=1, dash="dash"))

fig.show()
fig.write_image("figures/quantile_regression.pdf")

In [ ]:
# logistic

ts = []
for area in list(community_names.keys()):
    for spec in ['R+Y', 'R+P+Y', 'R+P+A+Y']:
        for r in range(5, 21, 5):
            print(area, community_names[area], r)
            t = data.query("phd_community == @area")[
                [
                    "PersonId",
                    "n",
                    "rao",
                    "gender",
                    "DegreeYear",
                    #"Percentile",
                    'phd_area',
                    "DegreePercentile",
                    #"DegreeInstitutionID",
                    f"rank{r}",
                    "cite_norm",
                    "adv_gender",
                    "seniority",
                    "adv_n",
                    'collab_num',
                ]
            ].drop_duplicates()
            
            # # use 0.1 rao as unit
            # t['rao'] = t['rao'] * 10
            
            # standardize
            t['rao'] = (t['rao'] - t['rao'].mean()) / t['rao'].std()
            
            # make fields as number
            t['phd_area'] = t['phd_area'].astype('category').cat.codes
            t['phd_area'] = t['phd_area'].astype('int')
            
            if spec == 'R+Y':
                control = "DegreePercentile i.DegreeYear" # 
            elif spec == 'R+P+Y':
                control = "DegreePercentile gender cite_norm n collab_num i.DegreeYear"
            elif spec == 'R+P+A+Y':
                control = "DegreePercentile gender cite_norm n collab_num adv_gender i.seniority adv_n i.DegreeYear"

            command = f"""
        logit rank{r} rao {control}, vce(robust) or
        """ # clogit rank{r} rao {control}, group(DegreeInstitutionID)

            stata.run("clear")
            stata.pdataframe_to_data(t)
            stata.run(command)

            # collect results
            results = stata.get_return()["r(table)"].T
            results = pd.DataFrame(results).iloc[0:1, :6]
            results.columns = ["coef", "se", "z", "p", "ci_low", "ci_high"]
            results["phd_community"] = area
            results["rank"] = r
            results["spec"] = spec
            ts.append(results)

ts = pd.concat(ts)
ts

In [16]:
ts['star'] = ts['p'].apply(lambda x: '＊＊＊' if x < 0.001 else '＊＊' if x < 0.01 else '＊' if x < 0.05 else '')
ts

,coef,se,z,p,ci_low,ci_high,phd_community,rank,spec,star
0,1.041928,0.078234,0.547012,0.584371,0.899341,1.207121,0,5,R+Y,
0,0.979079,0.054760,-0.378030,0.705408,0.877426,1.092509,0,10,R+Y,
0,1.027375,0.050483,0.549624,0.582577,0.933046,1.131242,0,15,R+Y,
0,0.977316,0.045009,-0.498226,0.618325,0.892964,1.069636,0,20,R+Y,
0,1.039171,0.080857,0.493818,0.621435,0.892187,1.210371,0,5,R+P+Y,
0,0.970593,0.055705,-0.520071,0.603014,0.867330,1.086150,0,10,R+P+Y,
0,1.019444,0.051079,0.384342,0.700725,0.924089,1.124639,0,15,R+P+Y,
0,0.968132,0.045416,-0.690382,0.489954,0.883087,1.061367,0,20,R+P+Y,
0,1.065128,0.086693,0.775189,0.438228,0.908072,1.249347,0,5,R+P+A+Y,
0,0.987872,0.060256,-0.200049,0.841443,0.876558,1.113321,0,10,R+P+A+Y,


In [ ]:
# plot coef line charts

fig = plot_coefplot(ts, xtitle="Odds ratio for standardized PhD interdisciplinarity predicting top faculty univ. placement")

fig.show()
fig.write_image("figures/logistic_reg_line_standardize.pdf")

In [23]:
# simple version (only 10% threshold)
# plot coef line charts

ts["Field"] = ts["phd_community"].map(community_names)
fig = make_subplots(
    rows=1,
    cols=5,
    shared_xaxes='all',
    shared_yaxes=True,
    subplot_titles=list(community_names.values()),
)
alpha = 0.8
colors = [
    f"rgba(33,142,33, {alpha})",
    f"rgba(133,205,235, {alpha})",
    f"rgba(255,165,0, {alpha})",
]

specs = ["R+Y", "R+P+Y", "R+P+A+Y"]
names = [
    "PhD univ. rank & grad year",
    "+ personal features",
    "+ advisor features (full control)",
]

tx = ts.query("rank == 10")
for i, area in enumerate(list(community_names.keys())):
    t2 = ts.query("phd_community == @area and rank == 10")
    for j, spec in enumerate(specs):
        t = t2.query("spec == @spec")
        x, y = t["coef"], [j]
        
        ci_high = t['ci_high']
        ci_low = t['ci_low']
        error_high = ci_high - x
        error_low = x - ci_low

        fig.add_trace(
            go.Scatter(
                x=x,
                y=y,
                error_x=dict(
                    type="data", symmetric=False, array=error_high, arrayminus=error_low,
                    width=3, thickness=1
                ),
                mode="markers",
                marker=dict(color=colors[j], size=5, opacity=1),
                name=names[j],
                showlegend= False,
                text=t["star"],
                hoverinfo="text",
            ),
            row=1,
            col=i % 5 + 1,
        )
        
        # add stars
        for r in [10]:
            t3 = t.query(f"rank == {r} and spec == @spec")
            star = t3["star"].values[0]
            fig.add_annotation(
                x=tx['ci_high'].max() + 0.0,
                y=y[0],
                yanchor="middle",
                yref="y",
                text=f"<b>{star}</b>",
                showarrow=False,
                font=dict(size=6, color=colors[j]),
                opacity=1,
                row=1,
                col=i % 5 + 1,
            )
            
        # add horizontal divider
        fig.add_hline(y=j + 0.5, line=dict(color="black", width=1), col=i % 5 + 1)

    # add shade left to 1
    fig.add_vrect(
        x0=tx['ci_low'].min() - 0.03,
        x1=1,
        fillcolor='lightgray',
        col=i % 5 + 1,
    )
    

plot_styling(fig, size=[200, 850])
fig.update_layout(
    legend=dict(
        title="Specification controls",
        orientation="h",
        yanchor="top",
        y=-0.3,
        yref="paper",
        xanchor="center",
        x=0.5,
        xref="paper",
    )
)
fig.update_yaxes(tickvals=[0, 1, 2], ticktext=names, side='top', col=1)
fig.update_yaxes(
    range=[2.5, -0.5], mirror=True
)
fig.update_xaxes(
    title=dict(
        text="Odds ratio for standardized PhD interdisciplinarity predicting top 10% faculty univ. placement",
        standoff=10,
    ),
    row=1,
    col=3,
)
fig.update_xaxes(dtick=0.1, minor=dict(dtick=0.05, ticklen=5), mirror=True)

fig.show()
fig.write_image("figures/logistic_reg_line_standardize_simple.pdf")

In [ ]:
# logistic
ts = []

for r in range(5, 21, 5):
    print(f"rank{r}")

    for area in list(community_names.keys()):
            print(area, community_names[area])
            t = data.query(f"phd_community == @area")[
                [
                    "PersonId",
                    "n",
                    "rao",
                    "gender",
                    "DegreeYear",
                    "Percentile",
                    "DegreePercentile",
                    f"rank{r}",
                    #f"degree_rank{r}",
                    "cite_norm",
                    "adv_gender",
                    "seniority",
                    "adv_n",
                    "collab_num"
                ]
            ].drop_duplicates()
                        
            control = "gender cite_norm n collab_num adv_gender i.seniority adv_n"

            command = f"""
        qui logit rank{r} c.rao##c.DegreePercentile {control}, vce(robust)
        margins, dydx(rao) at(DegreePercentile=(0(10)100))
        """

            stata.run("clear")
            stata.pdataframe_to_data(t)
            stata.run(command)

            # collect results
            results = stata.get_return()["r(table)"].T
            results = pd.DataFrame(results).iloc[:, :6]
            results.columns = ["coef", "se", "z", "p", "ci_low", "ci_high"]
            results["phd_community"] = area
            results["rank"] = r
            results["DegreePercentile"] = np.arange(0, 101, 10)
            
            ts.append(results)

ts = pd.concat(ts)

In [104]:
# plot
groupvar, group = "phd_community", list(community_names.keys())
titles = [name for name in community_names.values()]

fig = make_subplots(
    rows=4,
    cols=5,
    subplot_titles=titles,
    vertical_spacing=0.06,
    horizontal_spacing=0.06,
)

for i, domain in enumerate(group):
    for j, top in enumerate(range(5, 21, 5)):

        t1_domain = ts.query(f"{groupvar} == @domain and rank == @top")
        t1 = t1_domain.dropna()

        x = np.arange(0, 101, 10)
        y1 = t1["coef"].to_numpy()
        y1_upper, y1_lower = (
            t1["ci_low"].to_numpy(),
            t1["ci_high"].to_numpy(),
        )

        fig = add_lines_with_errorband(
            fig,
            x,
            y1,
            y1_upper,
            y1_lower,
            name='Margin',
            color=community_colors[domain],
            showlegend=False,
            row=j + 1,
            col=i % 5 + 1,
        )

fig = plot_styling(fig, size=(600, 800), title=None)
# update legend title
fig.update_layout(legend=dict(title="Faculty university"))

for row in range(1, 5):
    fig.update_yaxes(
        title=dict(text=f"X={row * 5}%", standoff=0),
        titlefont=dict(color="black"),
        tickfont=dict(color="black"),
        col=1,
        row=row,
    )

# zeroline
fig.add_hline(y=0, line=dict(dash="dash", width=1))

# Update layout to include super x and y labels
fig.update_xaxes(title=dict(text=None, standoff=0), dtick=20)
fig.add_annotation(
    text="PhD university rank percentile",
    x=0.5,
    y=-0.11,
    showarrow=False,
    xref="paper",
    xanchor="center",
    yref="paper",
    yanchor="bottom",
    font=dict(size=16),
)

fig.show()
fig.write_image("figures/phd_margin.pdf")

# IDR vs discipline mobility

In [15]:
numer = (
    df_univ_base[["PersonId", "phd_community", "rao_cut", "is_cross"]]
    .drop_duplicates()
    .groupby(["phd_community", "rao_cut", "is_cross"])
    .agg(n=("PersonId", "nunique"))
    .reset_index()
)
denom = (
    numer.groupby(["phd_community", "rao_cut"]).agg(total=("n", "sum")).reset_index()
)

numer = numer.merge(denom, on=["phd_community", "rao_cut"], how="left")
numer["pct"] = numer["n"] / numer["total"]
numer["Field"] = numer["phd_community"].map(community_names)
numer["Move type"] = numer["is_cross"].map({0: "Same field", 1: "Adjacent field", 2: "Distant field"})

colors = generate_gradient_colors(3)
print(colors)

# plot
fig = px.area(
    numer,
    x="rao_cut",
    y="pct",
    color="Move type",
    facet_col="Field",
    facet_col_wrap=5,
    facet_row_spacing=0.05,
    facet_col_spacing=0.05,
    color_discrete_map={k: v for k, v in zip(numer["Move type"].unique(), colors)},
    labels=dict(pct="% faculty members", rao_cut=""),
)

plot_styling(fig, size=[250, 800])
fig.update_layout(legend_title_text=None, title=None)
fig.update_xaxes(title_text="PhD interdisciplinarity", row=1, col=3)
fig.update_yaxes(title_text="% faculty members", tickformat=".0%", row=1, col=1)

# remove Field
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[1]))

fig.show()
#fig.write_image("figures/move_type_pct.pdf")

['rgb(91, 200, 98)', 'rgb(32, 143, 140)', 'rgb(59, 81, 138)']


In [ ]:
# average
t = (
    df_univ_base[["PersonId", "phd_community", "rao", "is_cross"]]
    .drop_duplicates()
    .groupby(["phd_community", "is_cross"])
    .agg(n=("rao", "mean"), confint=("rao", calculate_confidence_interval))
    .reset_index()
)

# map
t['is_cross'] = t['is_cross'].map({0: "Same field", 1: "Adjacent field", 2: "Distant field"})
t['phd_community'] = t['phd_community'].map(community_names)

# plot
fig = px.bar(
    t,
    x="phd_community",
    y="n",
    color="is_cross",
    error_y="confint",
    barmode="group",
    color_discrete_map={k: v for k, v in zip(t['is_cross'].unique(), colors)},
    labels=dict(n="Average interdisciplinarity", is_cross="Move type"),
)

plot_styling(fig, size=[300, 600])
fig.update_layout(
    legend_title_text=None,
    title=None,
)
fig.update_yaxes(title_text="Average interdisciplinarity")
fig.update_xaxes(title=None)

fig.show()
#fig.write_image("figures/move_type_avg_idr.pdf")

In [18]:
for r in [5, 10]:
    rank = 'rank' + str(r)

    t = df_univ_base[['PersonId', 'is_cross', rank]].drop_duplicates()
    # combine close and distant
    t['is_cross'] = t['is_cross'].map({0: 0, 1: 1, 2: 1})
    
    # total, not divided by discipline
    numer = t.query(f"{rank} == 1").groupby(['is_cross']).agg(n=('PersonId', 'nunique')).reset_index()
    denom = numer['n'].sum()
    
    df_top = numer.copy()
    df_top['total'] = denom
    df_top['pct'] = df_top['n'] / df_top['total']
    df_top['rank'] = rank
    
    display(df_top)

# overall
numer = t.groupby(['is_cross']).agg(n=('PersonId', 'nunique')).reset_index()
denom = numer['n'].sum()
overall = numer.copy()
overall['n_y'] = denom
overall['pct'] = overall['n'] / overall['n_y']
overall['rank'] = 'Overall'
display(overall)

,is_cross,n,total,pct,rank
0,0,1439,2044,0.704012,rank5
1,1,605,2044,0.295988,rank5


,is_cross,n,total,pct,rank
0,0,3285,4526,0.725806,rank10
1,1,1241,4526,0.274194,rank10


,is_cross,n,n_y,pct,rank
0,0,24564,31501,0.779785,Overall
1,1,6937,31501,0.220215,Overall


In [19]:
ts = []

for r in [5, 10]:
    rank = 'rank' + str(r)

    t = df_univ_base[['PersonId', 'phd_community', 'rao', 'is_cross', rank]].drop_duplicates()

    # by discipline
    numer = t.query(f"{rank} == 1").groupby(['phd_community', 'is_cross']).agg(n=('PersonId', 'nunique')).reset_index()
    denom = numer.groupby(['phd_community']).agg(n=('n', 'sum')).reset_index()
    data = numer.merge(denom, on=['phd_community'])
    data['pct'] = data['n_x'] / data['n_y']
    data['rank'] = f'Top {r}%'
    
    ts.append(data)

# overall
numer = t.groupby(['phd_community', 'is_cross']).agg(n=('PersonId', 'nunique')).reset_index()
denom = numer.groupby(['phd_community']).agg(n=('n', 'sum')).reset_index()
overall = numer.merge(denom, on=['phd_community'])
overall['pct'] = overall['n_x'] / overall['n_y']
overall['rank'] = 'Overall'
ts.append(overall)

data = pd.concat(ts)

data['Field'] = data['phd_community'].apply(lambda x: community_names[x])
data['Move type'] = data['is_cross'].map({0: "same-field", 1: "close-field", 2: "distant-field"})
data

,phd_community,is_cross,n_x,n_y,pct,rank,Field,Move type
0,0,0,238,310,0.767742,Top 5%,Math & Computing,same-field
1,0,1,55,310,0.177419,Top 5%,Math & Computing,close-field
2,0,2,17,310,0.054839,Top 5%,Math & Computing,distant-field
3,1,0,305,414,0.736715,Top 5%,Phys & Eng,same-field
4,1,1,90,414,0.217391,Top 5%,Phys & Eng,close-field
5,1,2,19,414,0.045894,Top 5%,Phys & Eng,distant-field
6,2,0,142,205,0.692683,Top 5%,Life & Earth,same-field
7,2,1,33,205,0.160976,Top 5%,Life & Earth,close-field
8,2,2,30,205,0.146341,Top 5%,Life & Earth,distant-field
9,3,0,260,413,0.629540,Top 5%,Bio & Health,same-field


In [33]:
# Generate gradient colors for the bars
colors = generate_gradient_colors(3)
color_map = {k: v for k, v in zip(data['Move type'].unique(), colors)}
moves = ["same-field", "close-field", "distant-field"]
move_labels = ["Same field", "Adjacent field", "Distant field"]

# Create subplots
fig = make_subplots(
    rows=1, 
    cols=len(community_names), 
    subplot_titles=list(community_names.values()),
    shared_yaxes=True,
    horizontal_spacing=0.08,
)

# Add stacked bar traces for each field
for col_idx, field in enumerate(list(community_names.keys())):
    for move_type in moves:
        move_data = data.query("phd_community == @field and `Move type` == @move_type and rank != 'Top 5%'")
        fig.add_trace(
            go.Bar(
                x=move_data['rank'],
                y=move_data['pct'],
                name=move_labels[moves.index(move_type)],
                marker_color=color_map[move_type],
                showlegend=(col_idx == 1)  # Show legend only for the first subplot
            ),
            row=1,
            col=col_idx + 1
        )

# Update layout and axis styling
plot_styling(fig, size=[230, 600])
fig.update_layout(
    barmode="stack",
    bargap=0.3,
    legend_title_text="Placement type",
)

fig.update_xaxes(title=None, tickvals=[0, 1], ticktext=["Top 10% univ.", "All univ."])
fig.update_yaxes(tickformat=".0%", dtick=0.25, col=1)
fig.update_yaxes(title_text="% new faculty", row=1, col=1)
fig.update_annotations(font_size=13)

for col_idx in range(2, 6):
    fig.update_yaxes(showline=False, ticklen=0, col=col_idx)

fig.show()
fig.write_image("figures/move_type_bar.pdf")

In [13]:
# (next: control productivity)
df_prod_cite = pd.read_csv("person_highest_cite_prod.csv")
df_advisor = pd.read_csv("advisor_attr.csv")
df_collab = pd.read_csv("collab_num.csv", usecols=["PersonId", "collab_num"])

data = (
    df_univ_base.merge(df_prod_cite, on="PersonId", how="inner")
    .merge(df_advisor, on="PersonId", how="left")
    .merge(df_collab, on="PersonId", how="left")
    [
        [
            "PersonId",
            "phd_community",
            "prof_community",
            'phd_area',
            'prof_area',
            'DegreeYear',
            "rao",
            "rao_bin",
            "gender",
            "Percentile",
            "DegreePercentile",
            "degree_rank5",
            "degree_rank10",
            "degree_rank15",
            'degree_rank20',
            'rank5',
            'rank10',
            'rank15',
            'rank20',
            'DegreeInstitutionID',
            "cite_norm",
            "n",
            "adv_gender",
            'seniority', 'adv_n',
            'is_cross',
            'collab_num'
        ]
    ]
    .fillna({'cite_norm': 0, 'collab_num': 0})
    .drop_duplicates()
)
data['gender'] = data['gender'].apply(lambda x: 1 if x == 'F' else 0)
data['adv_gender'] = data['adv_gender'].apply(lambda x: 1 if x == 'F' else 0)


In [ ]:
# split subsamples
#groupvar, group = 'is_cross', [0, 1, 2]

# logistic
r = 20
ts = []
for area in list(community_names.keys()):
    #for g in group:
        print(area, community_names[area])
        t = data.query(f"phd_community == @area")[ #
            [
                "PersonId",
                "n",
                "rao",
                #"rao_bin",
                "gender",
                "DegreeYear",
                #"Percentile",
                "DegreePercentile",
                #"DegreeInstitutionID",
                f"rank{r}",
                #f"degree_rank{r}",
                "cite_norm",
                "adv_gender",
                "seniority",
                "adv_n",
                'is_cross',
                'collab_num'
            ]
        ].drop_duplicates()
        
        # cross rank and is_cross
        t['y'] = t.apply(lambda x: x['is_cross'] + 1 if x[f"rank{r}"] == 1 else 0, axis=1)
        # standardize
        t['rao'] = (t['rao'] - t['rao'].mean()) / t['rao'].std()
        
        control = "DegreePercentile gender cite_norm n collab_num adv_gender i.seniority adv_n i.DegreeYear"
        
        for outcome in range(t['y'].nunique()):

            command = f"""
        mlogit y rao {control}, vce(robust) baseoutcome(0)
        margins, dydx(rao) atmeans predict(pr outcome({outcome}))
        """

            stata.run("clear")
            stata.pdataframe_to_data(t)
            stata.run(command)

            # collect results
            results = stata.get_return()["r(table)"].T
            results = pd.DataFrame(results).iloc[0:1, :6]
            results.columns = ["coef", "se", "z", "p", "ci_low", "ci_high"]
            results["phd_community"] = area
            results["rank"] = r
            results["outcome"] = outcome
            ts.append(results)

ts = pd.concat(ts)
ts

In [21]:
ts = ts.applymap(lambda x: np.nan if x > 1000 else round(x, 5))
ts["Field"] = ts["phd_community"].apply(lambda x: community_names[x])
ts["Placement"] = ts["outcome"].apply(
    lambda x: (
        f"Top {r}% universities, distant field"
        if x == 3
        else (
            f"Top {r}% universities, adjacent field"
            if x == 2
            else (
                f"Top {r}% universities, same field"
                if x == 1
                else "Non-top universities"
            )
        )
    )
)

In [22]:
# plot bar plot
colors = ['gray'] + generate_gradient_colors(3, palette_name="viridis_r")

fig = make_subplots(
    rows=1,
    cols=5,
    shared_xaxes=False,
    shared_yaxes=False,
    subplot_titles=list(community_names.values()),
    horizontal_spacing=0.07,
)

for i, area in enumerate(list(community_names.keys())):
    for j in range(ts['outcome'].nunique()):
        t = ts.query("phd_community == @area and outcome == @j")
        
        # multiply by 100 to get percentage point change
        t['coef'] = t['coef'] * 100
        t['ci_low'] = t['ci_low'] * 100
        t['ci_high'] = t['ci_high'] * 100
        
        color = colors[j]
        error_high = (t['ci_high'] - t['coef'])
        error_low = t['coef'] - t['ci_low']

        fig.add_trace(
            go.Bar(
                x=t["Placement"],
                y=t["coef"],
                error_y=dict(type="data", symmetric=False, array=error_high, arrayminus=error_low, thickness=1, width=0),
                marker=dict(color=color),
                name=t['Placement'].iloc[0],
                showlegend=True if i == 4 else False,
            ),
            row=1,
            col=i + 1,
        )
    
plot_styling(fig, size=[250, 800])
fig.update_layout(
    legend=dict(
        title=None,
        orientation="h",
        yanchor="top",
        y=-0.1,
        yref="paper",
        xanchor="center",
        x=0.3,
        xref="paper",
    )
)
    
fig.update_xaxes(showline=True, showticklabels=False, ticklen=0, mirror=True)
fig.update_yaxes(title=dict(text="Marginal prob. change by<br>1 SD of <i>RS</i> increase<br>(percentage point)"), row=1, col=1)
fig.update_yaxes(zeroline=True, zerolinewidth=1, zerolinecolor='black', mirror=True, tickformat=".1f")
fig.show()
fig.write_image(f"figures/logistic_reg_bar_top{r}_standardize.pdf")

In [ ]:
# split subsamples
#groupvar, group = 'is_cross', [0, 1, 2]

# logistic
r = 5
ts = []
for area in list(community_names.keys()):
    #for g in group:
        print(area, community_names[area])
        t = data.query(f"phd_community == @area")[ #
            [
                "PersonId",
                "n",
                "rao",
                #"rao_bin",
                "gender",
                "DegreeYear",
                #"Percentile",
                "DegreePercentile",
                #"DegreeInstitutionID",
                f"rank{r}",
                #f"degree_rank{r}",
                "cite_norm",
                "adv_gender",
                "seniority",
                "adv_n",
                'is_cross',
                'collab_num'
            ]
        ].drop_duplicates()
        
        control = "DegreePercentile gender cite_norm n collab_num adv_gender i.seniority adv_n i.DegreeYear"
        
        for outcome in range(t['is_cross'].nunique()):

            command = f"""
        logit rank{r} rao {control} if is_cross=={outcome}, vce(robust)
        """

            stata.run("clear")
            stata.pdataframe_to_data(t)
            stata.run(command)

            # collect results
            results = stata.get_return()["r(table)"].T
            results = pd.DataFrame(results).iloc[0:1, :6]
            results.columns = ["coef", "se", "z", "p", "ci_low", "ci_high"]
            results["phd_community"] = area
            results["rank"] = r
            results["outcome"] = outcome
            ts.append(results)

ts = pd.concat(ts)
ts

In [ ]:
# use move type as X

ts = []

for r in range(5, 21, 5):
    for area in list(community_names.keys()):
        #for g in group:
            print(area, community_names[area])
            t = data.query(f"phd_community == @area")[ #
                [
                    "PersonId",
                    "n",
                    "rao",
                    #"rao_bin",
                    "gender",
                    "DegreeYear",
                    #"Percentile",
                    "DegreePercentile",
                    #"DegreeInstitutionID",
                    f"rank{r}",
                    #f"degree_rank{r}",
                    "cite_norm",
                    "adv_gender",
                    "seniority",
                    "adv_n",
                    'is_cross',
                    'collab_num'
                ]
            ].drop_duplicates()
            
            control = "DegreePercentile gender cite_norm n collab_num adv_gender i.seniority adv_n i.DegreeYear"
            
            command = f"logit rank{r} i.is_cross {control}, vce(robust)"

            stata.run("clear")
            stata.pdataframe_to_data(t)
            stata.run(command)

            # collect results
            results = stata.get_return()["r(table)"].T
            results = pd.DataFrame(results).iloc[1:3, :6]
            results.columns = ["coef", "se", "z", "p", "ci_low", "ci_high"]
            results["phd_community"] = area
            results["rank"] = r
            results["var"] = ['close-field', 'distant-field']
            ts.append(results)

ts = pd.concat(ts)
ts

In [ ]:
import copy

# use stata to do psm
no_psm = pd.DataFrame()
no_rao = pd.DataFrame()
with_rao = pd.DataFrame()

controls = "DegreePercentile cite_norm n collab_num i.adv_gender i.seniority adv_n i.DegreeYear"

for area in list(community_names.keys()):
    print(community_names[area])
    t = data.query("phd_community == @area")
    
    stata.run("clear")
    stata.pdataframe_to_data(t)
    # get mean, ci
    stata.run('logit rank10 i.gender, or')
    
    # collect results
    results = stata.get_return()["r(table)"].T
    results = pd.DataFrame(results).iloc[1:2, :6]
    results.columns = ["coef", "se", "z", "p", "ci_low", "ci_high"]
    results["phd_community"] = area
    no_psm = pd.concat([no_psm, results], ignore_index=True)
    
    # psm
    stata.run("clear")
    stata.pdataframe_to_data(t)
    stata.run(f"qui psmatch2 gender {controls}, outcome(rank10) ate radius caliper(0.1) ties logit common")
    
    # get mean, ci weighted by _weight
    stata.run('logit rank10 i.gender [pweight=_weight] if _support==1, or'
              )
    
    # collect results
    results = stata.get_return()["r(table)"].T
    results = pd.DataFrame(results).iloc[1:2, :6]
    results.columns = ["coef", "se", "z", "p", "ci_low", "ci_high"]
    results["phd_community"] = area
    no_rao = pd.concat([no_rao, results], ignore_index=True)
    
    stata.run("clear")
    stata.pdataframe_to_data(t)
    # psm
    stata.run(f"qui psmatch2 gender rao {controls}, outcome(rank10) ate radius caliper(0.1) ties logit common")
    # get mean, ci weighted by _weight
    stata.run('logit rank10 i.gender [pweight=_weight] if _support==1, or'
              )
    
    # collect results
    results = stata.get_return()["r(table)"].T
    results = pd.DataFrame(results).iloc[1:2, :6]
    results.columns = ["coef", "se", "z", "p", "ci_low", "ci_high"]
    results["phd_community"] = area
    with_rao = pd.concat([with_rao, results], ignore_index=True)
    

In [ ]:
# balance check table

import copy

# use stata to do psm
no_psm = pd.DataFrame()
no_rao = pd.DataFrame()
with_rao = pd.DataFrame()

controls = "DegreePercentile cite_norm n collab_num i.adv_gender i.seniority adv_n i.DegreeYear"

for area in list(community_names.keys()):
    print(community_names[area])
    t = data.query("phd_community == @area")
    
    # psm
    stata.run("clear")
    stata.pdataframe_to_data(t)
    stata.run(f"qui psmatch2 gender {controls}, outcome(rank10) ate radius caliper(0.1) ties logit common")
    
    # check
    stata.run('pstest, both')
    
    stata.run("clear")
    stata.pdataframe_to_data(t)
    # psm
    stata.run(f"qui psmatch2 gender rao {controls}, outcome(rank10) ate radius caliper(0.1) ties logit common")
    # check
    stata.run('pstest, both')
    

In [46]:
t1 = no_rao
t2 = with_rao
t3 = no_psm
t1['type'] = 'no_rao'
t2['type'] = 'with_rao'
t3['type'] = 'no_psm'
t_compare = pd.concat([t1, t2, t3])
# sort by type
order = ['with_rao', 'no_rao', 'no_psm']
t_compare['order'] = t_compare['type'].apply(lambda x: order.index(x))
t_compare = t_compare.sort_values('order')
# p star
t_compare['star'] = t_compare['p'].apply(lambda x: '＊＊＊' if x < 0.001 else '＊＊' if x < 0.01 else '＊' if x < 0.05 else '')

# plot
fig = make_subplots(
    rows=1,
    cols=5,
    shared_yaxes="all",
    subplot_titles=list(community_names.values()),
)

xlabels = ["w/ PhD id.", "w/o PhD id.", "Original"]

for i, area in enumerate(list(community_names.keys())):
    t = t_compare.query("phd_community == @area")
    x = t['order']
    y = t["coef"]
    error_high = t['ci_high'] - y
    error_low = y - t['ci_low']
    
    color = gender_colors["M"]
    fig.add_trace(
        go.Scatter(
            x=x,
            y=y,
            error_y=dict(type="data", symmetric=False, array=error_high, arrayminus=error_low, width=0, thickness=1),
            mode="markers+lines",
            marker=dict(color=color),
            line=dict(color=color, width=1),
        ),
        row=i // 5 + 1,
        col=i % 5 + 1,
    )

    # add star as annotation
    for j, row in t.iterrows():
        fig.add_annotation(
            x=row['order'],
            y=row["ci_high"] + 0.05,
            text=row["star"],
            showarrow=False,
            font=dict(size=10, color=color),
            row=i // 5 + 1,
            col=i % 5 + 1,
        )

plot_styling(fig, size=[230, 980])
fig.update_layout(showlegend=False)
fig.update_xaxes(tickvals=[0, 1, 2], ticktext=xlabels)
fig.update_yaxes(dtick=0.5, minor=dict(dtick=0.1, ticklen=2), mirror=True)
fig.update_yaxes(title_text="Odds ratio<br>Placed at top 10% univ.<br>(women over men)", row=1, col=1)
fig.update_xaxes(mirror=True)

# add ref line
fig.add_hline(y=1, line=dict(color="black", width=1, dash="dash"))

fig.show()
fig.write_image("figures/psm_gender.pdf")

In [14]:
# gender ratio among top x% universities
datas = []
for x in np.arange(5, 105, 5):
    t = df_univ_base.query("Percentile >= 100 - @x")

    # each gender
    data = (
        t.groupby(["phd_community", "gender"])
        .agg(
            n=("PersonId", "nunique"),
        )
        .reset_index()
    )

    # get sum
    t_sum = (
        data.groupby(
            [
                "phd_community",
            ]
        )
        .agg(total=("n", "sum"))
        .reset_index()
    )
    data = data.merge(t_sum, on=["phd_community"], how="inner")
    data["share"] = data["n"] / data["total"]

    data["x"] = x

    datas.append(data)

data = pd.concat(datas)
data

,phd_community,gender,n,total,share,x
0,0,F,69,309,0.223301,5
1,0,M,240,309,0.776699,5
2,1,F,109,412,0.264563,5
3,1,M,303,412,0.735437,5
4,2,F,73,202,0.361386,5
...,...,...,...,...,...,...
5,2,M,2682,4036,0.664519,100
6,3,F,3779,7208,0.524279,100
7,3,M,3429,7208,0.475721,100
8,4,F,4998,10332,0.483740,100


In [15]:
# plot
fig = make_subplots(
    rows=1,
    cols=5,
    shared_xaxes=False,
    shared_yaxes=True,
    subplot_titles=list(community_names.values()),
)
for i, area in enumerate(list(community_names.keys())):
    for j, gender in enumerate(["M", "F"]):
        t = data.query("phd_community == @area and gender == @gender")
        color = gender_colors[gender]
        fig.add_trace(
            go.Scatter(
                x=t["x"],
                y=t["share"],
                mode="lines",
                name='Woman' if gender == 'F' else 'Man',
                marker=dict(color=color),
                showlegend=True if i == 0 else False,
            ),
            row=1,
            col=i + 1,
        )

plot_styling(fig, size=[240, 800])
fig.update_xaxes(showgrid=True)
fig.update_xaxes(title_text="Top X% faculty universities", col=3)
fig.update_yaxes(tickformat='.0%', dtick=0.2, range=(0, 1))
fig.update_yaxes(title_text="Cumulative share", row=1, col=1)

fig.show()
fig.write_image("figures/faculty_gender_share.pdf")